In [ ]:
!pip install -q --upgrade "transformers>=4.49.0"
import os
print("🔄 Restarting kernel to activate the new transformers version...")
os._exit(0)

In [2]:
import gc
import sys
import time
from unittest.mock import MagicMock
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import HTML, display

# 1. Hardware setup & data types
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (
    torch.float16 if torch.cuda.is_available() else torch.float32
)

BASE_MODEL_ID = "Qwen/Qwen3.5-0.8B-Base"
CPT_MODEL_ID = "kaptaan45/QaptaanLM-0.75B"

TEST_PROMPTS = [
    {
        "category": "Python Code Completion",
        "title": "Two Sum with Indices",
        "prompt": 'def two_sum(nums: list[int], target: int) -> list[int]:\n    """Return indices of two numbers that add up to target."""\n',
    },
    {
        "category": "Algorithmic Logic",
        "title": "Reverse Singly Linked List",
        "prompt": 'class ListNode:\n    def __init__(self, val=0, next=None):\n        self.val = val\n        self.next = next\n\ndef reverse_list(head: ListNode) -> ListNode:\n    """Reverses a singly linked list in-place and returns the new head."""\n',
    },
    {
        "category": "Fast Computation (Numpy)",
        "title": "Cosine Similarity Matrix",
        "prompt": 'import numpy as np\n\ndef batch_cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:\n    """Compute pair-wise cosine similarity between two 2D arrays a (N, D) and b (M, D)."""\n',
    },
    {
        "category": "Math Reasoning (Chain-of-Thought)",
        "title": "Word Problem",
        "prompt": "Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?\nAnswer: Let's think step by step.\n",
    },
    {
        "category": "Docstring to Implementation",
        "title": "Binary Search",
        "prompt": 'def binary_search(arr: list[int], target: int) -> int:\n    """Return index of target in sorted arr, or -1 if not found."""\n',
    }
]

def run_benchmark(model_id: str, prompts: list, max_new_tokens: int = 140):
    print(f"\n========================================================")
    print(f"Loading & Evaluating: {model_id}")
    print(f"========================================================")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=DTYPE,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()

    results = []
    for idx, item in enumerate(prompts):
        inputs = tokenizer(item["prompt"], return_tensors="pt").to(DEVICE)
        start_t = time.perf_counter()
        with torch.no_grad():
            output_tokens = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.6,
                top_p=0.9,
                repetition_penalty=1.15,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        elapsed = time.perf_counter() - start_t
        gen_tokens = output_tokens[0][inputs["input_ids"].shape[1]:]
        tok_count = len(gen_tokens)
        speed = tok_count / elapsed if elapsed > 0 else 0
        completion = tokenizer.decode(gen_tokens, skip_special_tokens=True)

        results.append({
            "category": item["category"],
            "title": item["title"],
            "prompt": item["prompt"],
            "completion": completion.strip(),
            "tokens_gen": tok_count,
            "speed": f"{speed:.1f} tok/s",
            "time_sec": round(elapsed, 2)
        })
        print(f"  [{idx+1}/{len(prompts)}] {item['title']}: {tok_count} tokens in {elapsed:.2f}s ({speed:.1f} tok/s)")

    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results

# Run Base & CPT Models
base_res = run_benchmark(BASE_MODEL_ID, TEST_PROMPTS)
cpt_res = run_benchmark(CPT_MODEL_ID, TEST_PROMPTS)

# Render Head-to-Head Comparison Table
html_output = "<h2>🚀 Head-to-Head Comparison (Fast Inference Enabled)</h2>"
for idx, (b, c) in enumerate(zip(base_res, cpt_res)):
    html_output += f"""
    <div style="border: 1px solid #334155; border-radius: 8px; margin-bottom: 20px; padding: 16px; background-color: #1e293b; color: #f8fafc;">
        <h3 style="color: #38bdf8; margin: 0 0 8px 0;">#{idx+1}. [{b['category']}] {b['title']}</h3>
        <pre style="background: #0f172a; padding: 8px; border-radius: 4px; color: #cbd5e1; font-size: 12px;">{b['prompt']}</pre>
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-top: 12px;">
            <div style="background: #0f172a; padding: 12px; border-radius: 6px; border-left: 3px solid #60a5fa;">
                <h4 style="color: #60a5fa; margin: 0 0 4px 0;">Base Model (Qwen3.5-0.8B)</h4>
                <small style="color: #94a3b8;">{b['tokens_gen']} tokens | {b['speed']} | {b['time_sec']}s</small>
                <pre style="background: #020617; padding: 8px; border-radius: 4px; color: #cbd5e1; white-space: pre-wrap; font-size: 12px; margin-top: 8px;">{b['completion']}</pre>
            </div>
            <div style="background: #0f172a; padding: 12px; border-radius: 6px; border-left: 3px solid #4ade80;">
                <h4 style="color: #4ade80; margin: 0 0 4px 0;">CPT Model (QaptaanLM-0.75B)</h4>
                <small style="color: #94a3b8;">{c['tokens_gen']} tokens | {c['speed']} | {c['time_sec']}s</small>
                <pre style="background: #020617; padding: 8px; border-radius: 4px; color: #4ade80; white-space: pre-wrap; font-size: 12px; margin-top: 8px;">{c['completion']}</pre>
            </div>
        </div>
    </div>
    """
display(HTML(html_output))


Loading & Evaluating: Qwen/Qwen3.5-0.8B-Base


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

  [1/5] Two Sum with Indices: 105 tokens in 6.83s (15.4 tok/s)
  [2/5] Reverse Singly Linked List: 95 tokens in 6.11s (15.6 tok/s)
  [3/5] Cosine Similarity Matrix: 74 tokens in 4.78s (15.5 tok/s)
  [4/5] Word Problem: 140 tokens in 8.94s (15.7 tok/s)
  [5/5] Binary Search: 140 tokens in 8.99s (15.6 tok/s)

Loading & Evaluating: kaptaan45/QaptaanLM-0.75B


Loading weights:   0%|          | 0/321 [00:00<?, ?it/s]

  [1/5] Two Sum with Indices: 140 tokens in 7.90s (17.7 tok/s)
  [2/5] Reverse Singly Linked List: 140 tokens in 8.04s (17.4 tok/s)
  [3/5] Cosine Similarity Matrix: 140 tokens in 8.01s (17.5 tok/s)
  [4/5] Word Problem: 140 tokens in 7.99s (17.5 tok/s)
  [5/5] Binary Search: 140 tokens in 7.94s (17.6 tok/s)


In [3]:
import gc
import json
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (
    torch.float16 if torch.cuda.is_available() else torch.float32
)

CODING_PROBLEMS = [
    {
        "task_id": "HumanEval/0",
        "prompt": 'def has_close_elements(numbers: list[float], threshold: float) -> bool:\n    """Check if in given list of numbers, are any two numbers closer to each other than given threshold."""\n',
        "test": "assert has_close_elements([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True\nassert has_close_elements([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05) == False\n",
    },
    {
        "task_id": "HumanEval/2",
        "prompt": 'def truncate_number(number: float) -> float:\n    """Return the decimal part of a positive floating point number."""\n',
        "test": "assert abs(truncate_number(3.5) - 0.5) < 1e-4\nassert abs(truncate_number(1.33) - 0.33) < 1e-4\n",
    },
    {
        "task_id": "HumanEval/8",
        "prompt": 'def sum_product(numbers: list[int]) -> tuple[int, int]:\n    """Return a tuple consisting of a sum and a product of all integers in a list."""\n',
        "test": "assert sum_product([]) == (0, 1)\nassert sum_product([1, 2, 3, 4]) == (10, 24)\n",
    },
    {
        "task_id": "HumanEval/11",
        "prompt": 'def string_xor(a: str, b: str) -> str:\n    """Perform binary XOR on two binary strings and return result as string."""\n',
        "test": "assert string_xor('111000', '101010') == '010010'\nassert string_xor('1', '1') == '0'\n",
    },
    {
        "task_id": "MBPP/1",
        "prompt": 'def min_cost(cost: list[list[int]], m: int, n: int) -> int:\n    """Find the minimum cost path to reach (m, n) from (0, 0) for given cost matrix."""\n',
        "test": "cost = [[1, 2, 3], [4, 8, 2], [1, 5, 3]]\nassert min_cost(cost, 2, 2) == 8\n",
    },
    {
        "task_id": "MBPP/3",
        "prompt": 'def is_not_prime(n: int) -> bool:\n    """Write a function to identify non-prime numbers."""\n',
        "test": "assert is_not_prime(2) == False\nassert is_not_prime(10) == True\nassert is_not_prime(35) == True\n",
    }
]

def test_code_snippet(prompt: str, completion: str, test_code: str) -> bool:
    code = completion.split("```python")[-1].split("```")[0] if "```" in completion else completion
    full_program = f"from typing import List, Tuple, Dict, Optional, Any, Union\nimport math\n\n{prompt}{code}\n\n{test_code}"
    try:
        exec(full_program, {}, {})
        return True
    except Exception:
        return False

def evaluate_coding(model_id: str):
    print(f"\n========================================================")
    print(f"Running Coding Benchmark (pass@1): {model_id}")
    print(f"========================================================")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True)
    model.eval()

    passed = 0
    total_tokens = 0
    t0 = time.perf_counter()

    for item in tqdm(CODING_PROBLEMS, desc=model_id.split('/')[-1]):
        inputs = tokenizer(item["prompt"], return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=180,
                do_sample=False,  # greedy pass@1
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen = out[0][inputs["input_ids"].shape[1]:]
        total_tokens += len(gen)
        completion = tokenizer.decode(gen, skip_special_tokens=True)
        if test_code_snippet(item["prompt"], completion, item["test"]):
            passed += 1

    total_time = time.perf_counter() - t0
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "model_id": model_id,
        "pass_rate": (passed / len(CODING_PROBLEMS)) * 100,
        "passed": passed,
        "total": len(CODING_PROBLEMS),
        "throughput": total_tokens / total_time,
        "time_sec": total_time,
    }

base_code = evaluate_coding("Qwen/Qwen3.5-0.8B-Base")
cpt_code = evaluate_coding("kaptaan45/QaptaanLM-0.75B")

print("\n🏆 Coding Benchmark (pass@1) Results:")
print(f"  Base Model (Qwen3.5-0.8B):  {base_code['passed']}/{base_code['total']} ({base_code['pass_rate']:.1f}%) | {base_code['throughput']:.1f} tok/s")
print(f"  CPT Model (QaptaanLM-0.75B): {cpt_code['passed']}/{cpt_code['total']} ({cpt_code['pass_rate']:.1f}%) | {cpt_code['throughput']:.1f} tok/s")


Running Coding Benchmark (pass@1): Qwen/Qwen3.5-0.8B-Base


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Qwen3.5-0.8B-Base:   0%|          | 0/6 [00:00<?, ?it/s]


Running Coding Benchmark (pass@1): kaptaan45/QaptaanLM-0.75B


Loading weights:   0%|          | 0/321 [00:00<?, ?it/s]

QaptaanLM-0.75B:   0%|          | 0/6 [00:00<?, ?it/s]


🏆 Coding Benchmark (pass@1) Results:
  Base Model (Qwen3.5-0.8B):  2/6 (33.3%) | 15.8 tok/s
  CPT Model (QaptaanLM-0.75B): 0/6 (0.0%) | 18.4 tok/s


In [4]:
REASONING_DATA = {
    "MMLU (Computer Science & Physics)": [
        {"prompt": "Question: In computer networking, which OSI layer is responsible for end-to-end transport and flow control?\nA. Transport layer\nB. Network layer\nC. Data link layer\nD. Physical layer\nAnswer:", "correct": "A"},
        {"prompt": "Question: What is the derivative of f(x) = 4*x^3 with respect to x?\nA. 12x^2\nB. 4x^2\nC. 12x^3\nD. 7x^2\nAnswer:", "correct": "A"},
    ],
    "ARC-Challenge (Science QA)": [
        {"prompt": "Question: Which property of a mineral can be tested by scratching it against glass?\nA. Hardness\nB. Luster\nC. Streak\nD. Density\nAnswer:", "correct": "A"},
        {"prompt": "Question: What type of energy transformation occurs when a battery powers an LED light?\nA. Chemical to electrical to light\nB. Nuclear to radiant\nC. Thermal to sound\nD. Solar to kinetic\nAnswer:", "correct": "A"},
    ],
    "GSM8K (Math Word Problems)": [
        {"prompt": "Question: A train travels at 60 mph for 3 hours. How many miles does it travel in total?\nAnswer: Let's calculate step by step.", "correct": "180"},
        {"prompt": "Question: Anna bought 5 apples at $2 each and 3 oranges at $3 each. How much did she spend in total?\nAnswer: Let's calculate step by step.", "correct": "19"},
    ]
}

def evaluate_reasoning(model_id: str):
    print(f"\n========================================================")
    print(f"Running Reasoning Benchmark (Accuracy): {model_id}")
    print(f"========================================================")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True)
    model.eval()

    domain_scores = {}
    total_correct = 0
    total_samples = 0

    for domain, samples in REASONING_DATA.items():
        correct = 0
        for item in samples:
            inputs = tokenizer(item["prompt"], return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=60, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
            comp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
            if item["correct"] in comp or item["correct"].lower() in comp.lower():
                correct += 1
        acc = (correct / len(samples)) * 100
        domain_scores[domain] = f"{correct}/{len(samples)} ({acc:.1f}%)"
        total_correct += correct
        total_samples += len(samples)

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return {"overall_accuracy": (total_correct / total_samples) * 100, "domains": domain_scores}

base_reas = evaluate_reasoning("Qwen/Qwen3.5-0.8B-Base")
cpt_reas = evaluate_reasoning("kaptaan45/QaptaanLM-0.75B")

print("\n🧠 Reasoning Benchmark Results:")
for d in base_reas["domains"]:
    print(f"  {d:<40} | Base: {base_reas['domains'][d]:<12} | CPT: {cpt_reas['domains'][d]}")



Running Reasoning Benchmark (Accuracy): Qwen/Qwen3.5-0.8B-Base


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]


Running Reasoning Benchmark (Accuracy): kaptaan45/QaptaanLM-0.75B


Loading weights:   0%|          | 0/321 [00:00<?, ?it/s]


🧠 Reasoning Benchmark Results:
  MMLU (Computer Science & Physics)        | Base: 2/2 (100.0%) | CPT: 2/2 (100.0%)
  ARC-Challenge (Science QA)               | Base: 2/2 (100.0%) | CPT: 2/2 (100.0%)
  GSM8K (Math Word Problems)               | Base: 0/2 (0.0%)   | CPT: 0/2 (0.0%)
